In [6]:
!pip install transformers safetensors onnx

In [7]:
# ═════════════════════════════════════════════════════════════════════════════
# 📦 Imports
# ═════════════════════════════════════════════════════════════════════════════

import json
import os
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GPT2LMHeadModel,
    GPT2Tokenizer,
)
from transformers.onnx import export

# ═════════════════════════════════════════════════════════════════════════════
# 🔵 Full Auto-Export Script (Final Fixed Version)
# Vanilla + GRPO + SFT + PPO (from .bin) → ONNX
# Each model saved separately
# ═════════════════════════════════════════════════════════════════════════════

# --- SETTINGS ---
opset_version = 16  # Set to 16 like you wanted

# --- SAFETENSORS PATHS ---
safetensors_paths = {
    "grpo": "/content/grpo.safetensors",
    "sft": "/content/sft.safetensors",
}
config_json_path = "/content/config.json"

# --- PPO PYTORCH FILE PATH ---
ppo_model_file = "/content/pytorch_model.bin"

In [8]:
# ═════════════════════════════════════════════════════════════════════════════
# 🛠 Create a config.json for GPT-2 manually (for your GRPO and SFT safetensors)
# ═════════════════════════════════════════════════════════════════════════════

config = {
    "architectures": ["GPT2LMHeadModel"],
    "model_type": "gpt2",
    "n_embd": 768,
    "n_layer": 12,
    "n_head": 12,
    "vocab_size": 50257,
    "bos_token_id": 50256,
    "eos_token_id": 50256,
    "pad_token_id": 50256
}

# Save the config.json to /content
with open("/content/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("✅ Created /content/config.json successfully!")


✅ Created /content/config.json successfully!


In [9]:
# --- HELPER: prepare safetensors folder ---
def prepare_safetensor_folder(safetensor_file, config_file, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    os.system(f"cp {safetensor_file} {output_folder}/model.safetensors")
    os.system(f"cp {config_file} {output_folder}/config.json")

# --- MODELS TO EXPORT ---
models_to_export = {
    "vanilla": {
        "type": "hub",
        "save_path": "./gpt2_vanilla.onnx",
    },
    "grpo": {
        "type": "safetensor",
        "save_path": "./gpt2_grpo.onnx",
    },
    "sft": {
        "type": "safetensor",
        "save_path": "./gpt2_sft.onnx",
    },
    "ppo": {
        "type": "pytorch_bin",
        "save_path": "./gpt2_ppo.onnx",
    },
}

# --- START EXPORTING ---
for name, info in models_to_export.items():
    print(f"\n🔵 Processing: {name.upper()} model...")

    if info["type"] == "hub":
        model = AutoModelForCausalLM.from_pretrained("gpt2")
        tokenizer = AutoTokenizer.from_pretrained("gpt2")

    elif info["type"] == "safetensor":
        temp_folder = f"/content/{name}_temp_model_folder"
        prepare_safetensor_folder(
            safetensor_file=safetensors_paths[name],
            config_file=config_json_path,
            output_folder=temp_folder
        )
        model = AutoModelForCausalLM.from_pretrained(temp_folder, trust_remote_code=True)
        tokenizer = AutoTokenizer.from_pretrained("gpt2")  # use gpt2 tokenizer

    elif info["type"] == "pytorch_bin":
        # Manually create GPT2 model and load weights
        model = GPT2LMHeadModel.from_pretrained("gpt2")  # start from normal GPT2
        state_dict = torch.load(ppo_model_file, map_location="cpu")
        model.load_state_dict(state_dict, strict=False)  # load PPO .bin weights
        tokenizer = AutoTokenizer.from_pretrained("gpt2")  # use gpt2 tokenizer

    else:
        raise ValueError(f"Unknown model type for {name}")

    model.eval()

    dummy_input = tokenizer(
        "Hello, my name is",
        return_tensors="pt"
    )

    save_path = info["save_path"]

    # --- Export manually using torch.onnx.export ---
    torch.onnx.export(
        model,
        (dummy_input["input_ids"],),
        save_path,
        input_names=["input_ids"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "sequence"},
            "logits": {0: "batch_size", 1: "sequence"},
        },
        opset_version=opset_version,
        do_constant_folding=True,
    )

    print(f"✅ Exported {name.upper()} model to {save_path}")

print("\n🏁 All models exported successfully!")
# ═════════════════════════════════════════════════════════════════════════════


🔵 Processing: VANILLA model...
✅ Exported VANILLA model to ./gpt2_vanilla.onnx

🔵 Processing: GRPO model...
✅ Exported GRPO model to ./gpt2_grpo.onnx

🔵 Processing: SFT model...
✅ Exported SFT model to ./gpt2_sft.onnx

🔵 Processing: PPO model...
✅ Exported PPO model to ./gpt2_ppo.onnx

🏁 All models exported successfully!
